# M3 step 13: report tables (tasks 8 and 11)

Collects the results of notebooks 6 to 12 into the tables of section 6 of the handover guide and writes
them to `results/report_tables.md`, ready to paste into the report. Rows that have not been run yet
are marked "not run yet". Results from the dummy detector are never used.

**GPU:** not needed.

In [1]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

Working in /kaggle/working/VeriStamp/m3_experiments


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import settings as S
import m3_eval as E
R = S.RESULTS_DIR

def pct(v):
    return f'{100 * v:.2f}%'

def load(stem):
    f = R / f'{stem}_summary.csv'
    return pd.read_csv(f) if f.exists() else None

## RQ2: edits

In [3]:
NOT_RUN = ('not run yet',) * 3
import json
GUIDE = [('Saved as PNG', 'PNG', '99.85%', '96.0%'), ('JPEG quality 90', 'JPEG 90', '99.83%', '95.2%'),
         ('JPEG quality 70', 'JPEG 70', '99.58%', '90.0%'), ('JPEG quality 50', 'JPEG 50', '98.75%', '72.4%'),
         ('Resized to 224', 'Resized to 224', '99.83%', '95.4%')]
meta = R / 'setup_check_meta.json'
check = {r['edit']: r for r in json.loads(meta.read_text())['table']} if meta.exists() else {}
# From notebook 5's repeat of notebook 4, so the three columns of a row come from the same run; the
# guide's numbers (section 1) are used only if notebook 5 has not been run.
rows = [(label, f"{check[key]['bit_accuracy_%']:.2f}%", f"{check[key]['valid_verdict_%']:.1f}%",
         f"{check[key]['invalidation_rate_%']:.2f}%") if key in check else (label, b, v, 'run notebook 5')
        for label, key, b, v in GUIDE]

def take(table, edit, variant=None):
    if table is None:
        return NOT_RUN
    t = table[table.edit == edit]
    if variant is not None:
        t = t[t.variant == variant]
    if t.empty:
        return NOT_RUN
    r = t.iloc[0]
    return pct(r.bit_accuracy), pct(r.valid_verdict), pct(r.invalidation_rate)

shot, vae, inp = load('rq2_screenshot'), load('rq2_vae'), load('rq2_inpainting')
rows.append(('Screenshot simulation', *take(shot, 'Screenshot simulation (all steps)')))
rows.append(('VAE round trip', *take(vae, 'VAE round trip')))
for f in (10, 25, 50):
    rows.append((f'Inpainting, {f}% mask', *take(inp, f'Inpainting, {f}% mask', 'composited')))
rq2 = pd.DataFrame(rows, columns=['Edit', 'Bit accuracy', 'Valid verdict', 'Stamp invalidation rate'])
rq2

,Edit,Bit accuracy,Valid verdict,Stamp invalidation rate
0,Saved as PNG,99.85%,96.0%,run notebook 5
1,JPEG quality 90,99.83%,95.2%,run notebook 5
2,JPEG quality 70,99.58%,90.0%,run notebook 5
3,JPEG quality 50,98.75%,72.4%,run notebook 5
4,Resized to 224,99.83%,95.4%,run notebook 5
5,Screenshot simulation,not run yet,not run yet,not run yet
6,VAE round trip,not run yet,not run yet,not run yet
7,"Inpainting, 10% mask",not run yet,not run yet,not run yet
8,"Inpainting, 25% mask",not run yet,not run yet,not run yet
9,"Inpainting, 50% mask",not run yet,not run yet,not run yet


## Image quality

In [4]:
q = load('image_quality')
get = lambda name, fmt: fmt.format(q.loc[q.metric == name, 'mean'].iloc[0]) if q is not None else 'not run yet'
quality = pd.DataFrame([('PSNR', '31.50 dB, already measured' if q is None else get('PSNR (dB)', '{:.2f} dB')),
                        ('SSIM', get('SSIM', '{:.4f}')), ('LPIPS', get('LPIPS (AlexNet)', '{:.4f}'))],
                       columns=['Metric', 'Value on the 500 test photos'])
quality

,Metric,Value on the 500 test photos
0,PSNR,"31.50 dB, already measured"
1,SSIM,not run yet
2,LPIPS,not run yet


## RQ3: detectors

In [5]:
r3 = load('rq3_detectors')          # the _DUMMY file is ignored on purpose
names = {'m1_resnet': 'M1', 'm2_clip': 'M2', 'fusion': 'Late Fusion'}
rows = []
for det, label in names.items():
    t = None if r3 is None else r3[r3.detector == det]
    if t is None or t.empty:
        rows.append((label, 'not run yet', 'not run yet', 'not run yet', 'not run yet'))
        continue
    r = t.iloc[0]
    rows.append((label, f'{r.score_change_real:+.4f} [{r.ci95_real_low:+.4f}, {r.ci95_real_high:+.4f}]',
                 f'{r.score_change_ai:+.4f} [{r.ci95_ai_low:+.4f}, {r.ci95_ai_high:+.4f}]',
                 f'{int(r.verdict_flips)} of {int(r.n_real + r.n_ai)} ({int(r.real_to_ai)} Real to AI, '
                 f'{int(r.ai_to_real)} AI to Real, {int(r.into_uncertain + r.out_of_uncertain)} to or from Uncertain)', f'{pct(r.accuracy_before)} / {pct(r.accuracy_after)}'))
rq3 = pd.DataFrame(rows, columns=['Detector', 'Score change, real photos', 'Score change, AI photos',
                                  'Verdict flips', 'Accuracy before / after'])
rq3

,Detector,"Score change, real photos","Score change, AI photos",Verdict flips,Accuracy before / after
0,M1,"+0.0043 [-0.0061, +0.0144]","-0.0130 [-0.0249, -0.0015]","53 of 600 (0 Real to AI, 1 AI to Real, 52 to o...",67.50% / 66.67%
1,M2,"+0.0626 [+0.0478, +0.0795]","+0.0531 [+0.0314, +0.0748]","118 of 600 (3 Real to AI, 1 AI to Real, 114 to...",82.50% / 83.17%
2,Late Fusion,"+0.0335 [+0.0235, +0.0440]","+0.0200 [+0.0076, +0.0327]","72 of 600 (0 Real to AI, 0 AI to Real, 72 to o...",75.17% / 75.17%


## Extra tables: GenImage check and pipeline

In [6]:
g = load('genimage_check')
p = R / 'pipeline_summary.md'
extra = ''
if g is not None:
    extra += '### Stamp on GenImage photos\n\n' + E.to_markdown(E.as_percent(g)) + '\n\n'
if p.exists():
    extra += '### Pipeline\n\n' + p.read_text() + '\n'
print(extra or 'not run yet')

### Pipeline

| metric | value |
|---|---|
| photos | 400 |
| false skips (unstamped photo read as stamped) | 0 |
| AI photos stamped as Real | 45 of 200 (22.5%) |
| Real photos stamped as AI | 5 |
| Uncertain share | 24.8% |
| mean seconds per photo (first pass) | 0.0490 |
| returned as PNG: answered from stamp | 94.8% |
| returned as PNG: same verdict as first pass | 100.0% |
| returned after JPEG 90: answered from stamp | 90.2% |
| returned after JPEG 90: same verdict as first pass | 99.0% |
| returned after VAE round trip: answered from stamp | 15.8% |
| returned after VAE round trip: same verdict as first pass | 70.2% |

| class | Real | Uncertain | AI |
|---|---|---|---|
| AI | 45 | 79 | 76 |
| Real | 175 | 20 | 5 |




## Write `results/report_tables.md`

In [7]:
text = ('## RQ2: edits\n\n' + E.to_markdown(rq2) + '\n\n'
        'The first five rows repeat notebook 4 (notebook 5; the same as section 1 of the guide within 0.2 points).\n\n'
        '## Image quality\n\n' + E.to_markdown(quality) + '\n\n'
        '## RQ3: detectors\n\n' + E.to_markdown(rq3) + '\n\n' + extra)
(R / 'report_tables.md').write_text(text)
print(text)
if Path('/kaggle/working').is_dir():          # one file to download from the Output tab
    import shutil
    print('Download:', shutil.make_archive('/kaggle/working/m3_results', 'zip', R))

## RQ2: edits

| Edit | Bit accuracy | Valid verdict | Stamp invalidation rate |
|---|---|---|---|
| Saved as PNG | 99.85% | 96.0% | run notebook 5 |
| JPEG quality 90 | 99.83% | 95.2% | run notebook 5 |
| JPEG quality 70 | 99.58% | 90.0% | run notebook 5 |
| JPEG quality 50 | 98.75% | 72.4% | run notebook 5 |
| Resized to 224 | 99.83% | 95.4% | run notebook 5 |
| Screenshot simulation | not run yet | not run yet | not run yet |
| VAE round trip | not run yet | not run yet | not run yet |
| Inpainting, 10% mask | not run yet | not run yet | not run yet |
| Inpainting, 25% mask | not run yet | not run yet | not run yet |
| Inpainting, 50% mask | not run yet | not run yet | not run yet |

The first five rows repeat notebook 4 (notebook 5; the same as section 1 of the guide within 0.2 points).

## Image quality

| Metric | Value on the 500 test photos |
|---|---|
| PSNR | 31.50 dB, already measured |
| SSIM | not run yet |
| LPIPS | not run yet |

## RQ3: detectors

| Detector | Score cha